# Advanced LLM Training on Google Colab

Train a **300M parameter** hybrid Mamba-2 + Transformer + MoE model on Colab.

**Architecture:**
- 10 Mamba-2 SSM blocks + 2 GQA attention blocks (7:1 ratio)
- 8 routed + 1 shared MoE experts (DeepSeek-V3 style)
- RMSNorm, SwiGLU, RoPE, Flash Attention

**Requirements:** Colab with GPU runtime (T4 or A100 recommended)

## 1. Setup Environment

In [ ]:
!nvidia-smi
!python --version

In [ ]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install transformers datasets tokenizers bitsandbytes accelerate wandb pyyaml safetensors tqdm

In [ ]:
# Clone the project (or mount Google Drive)
import os
if not os.path.exists('/content/advanced-llm'):
    # Option 1: Clone from GitHub
    # !git clone https://github.com/YOUR_USERNAME/advanced-llm.git /content/advanced-llm

    # Option 2: Upload manually
    from google.colab import files
    print('Upload your advanced-llm project as a zip file:')
    uploaded = files.upload()
    for fname in uploaded:
        if fname.endswith('.zip'):
            !unzip -q {fname} -d /content/

os.chdir('/content/advanced-llm')
!ls -la

## 2. Download & Tokenize Data

In [ ]:
# Download FineWeb-Edu and train tokenizer
!python scripts/download_data.py \
    --dataset HuggingFaceFW/fineweb-edu \
    --subset sample-10BT \
    --num-samples 50000 \
    --train-tokenizer \
    --vocab-size 32000 \
    --output data/train_tokens.pt \
    --tokenizer-path tokenizer.json

## 3. Configure Training

Adjust parameters based on your GPU. T4 (16GB) needs smaller batch; A100 (40GB) can go bigger.

In [ ]:
import torch

gpu_name = torch.cuda.get_device_name() if torch.cuda.is_available() else 'CPU'
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9 if torch.cuda.is_available() else 0
print(f'GPU: {gpu_name}, Memory: {gpu_mem:.1f} GB')

# Auto-adjust config based on GPU
if gpu_mem >= 35:  # A100 40GB
    batch_size = 4
    grad_accum = 8
    seq_len = 4096
    max_steps = 50000
elif gpu_mem >= 15:  # T4 16GB
    batch_size = 2
    grad_accum = 16
    seq_len = 2048
    max_steps = 30000
else:  # Smaller GPU
    batch_size = 1
    grad_accum = 32
    seq_len = 1024
    max_steps = 20000

print(f'Config: batch={batch_size}, accum={grad_accum}, seq_len={seq_len}, steps={max_steps}')
print(f'Effective batch: {batch_size * grad_accum * seq_len:,} tokens')

## 4. Train the Model

In [ ]:
import sys
sys.path.insert(0, 'src')

import torch
from torch.utils.data import DataLoader
from model.config import ModelConfig
from model.transformer import HybridModel
from training.data import PreTokenizedDataset
from training.trainer import Trainer

# Model config (300M params)
model_config = ModelConfig(
    d_model=768,
    n_layers=12,
    vocab_size=32000,
    max_seq_len=seq_len,
    attention_layer_indices=[7, 11],
    d_state=64,
    d_conv=4,
    expand=2,
    n_heads=12,
    n_kv_heads=4,
    head_dim=64,
    num_experts=8,
    num_shared_experts=1,
    moe_top_k=2,
    expert_dim=2048,
)

model = HybridModel(model_config)
params = model.count_parameters()
print(f'Model: {params["total"]:,} total params, {params["trainable"]:,} trainable')
print(f'Memory estimate: {params["total"] * 2 / 1e9:.2f} GB (bf16 weights only)')

In [ ]:
# Load dataset
import os

data_path = 'data/train_tokens.pt'
if not os.path.exists(data_path):
    print(f'Data not found at {data_path}. Run the data download cell first.')
else:
    dataset = PreTokenizedDataset(data_path, seq_len)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
    )
    print(f'Dataset: {len(dataset):,} samples, {len(dataloader):,} batches')

In [ ]:
# Train
trainer = Trainer(
    model=model,
    train_dataloader=dataloader,
    lr=3e-4,
    weight_decay=0.1,
    warmup_steps=500,
    stable_steps=max_steps - 1000,
    decay_steps=500,
    grad_accum_steps=grad_accum,
    max_grad_norm=1.0,
    checkpoint_dir='checkpoints',
    log_interval=50,
    save_interval=5000,
    device='cuda',
    use_amp=True,
    optimizer_name='adamw8bit',
)

print(f'Starting training for {max_steps} steps...')
print(f'Estimated time: {max_steps * 0.5 / 3600:.1f} - {max_steps * 1.0 / 3600:.1f} hours')
history = trainer.train(max_steps=max_steps)
print(f'Training complete! Final loss: {history["train_loss"][-1]:.4f}')

## 5. Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'])
ax1.set_xlabel('Log Step')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.grid(True)

ax2.plot(history['lr'])
ax2.set_xlabel('Log Step')
ax2.set_ylabel('Learning Rate')
ax2.set_title('Learning Rate Schedule')
ax2.grid(True)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

## 6. Generate Text

In [ ]:
from inference.generate import generate
from training.tokenizer import load_tokenizer

# Load best model
from training.checkpoint import load_checkpoint
if os.path.exists('checkpoints/best_model.pt'):
    load_checkpoint(model, path='checkpoints/best_model.pt')
    print('Loaded best model checkpoint')

model.eval()

# Load tokenizer
tokenizer = load_tokenizer('tokenizer.json')

# Generate
prompts = [
    "The future of artificial intelligence",
    "Once upon a time in a land far away",
    "The scientific method involves",
]

for prompt in prompts:
    input_ids = torch.tensor([tokenizer.encode(prompt).ids], device='cuda')
    output_ids = generate(model, input_ids, max_new_tokens=100, temperature=0.8, top_p=0.9)
    text = tokenizer.decode(output_ids[0].tolist())
    print(f'\n--- Prompt: {prompt} ---')
    print(text)
    print()

## 7. Export & Download Model

In [ ]:
from inference.export import export_safetensors

# Export to SafeTensors
export_path = export_safetensors(model, 'export')
print(f'Exported to: {export_path}')

# List exported files
!ls -lh export/

In [ ]:
# Download the model
from google.colab import files

# Zip everything
!zip -r model_export.zip export/ tokenizer.json configs/

print('Downloading model_export.zip...')
files.download('model_export.zip')

## 8. (Optional) Save to Google Drive

In [ ]:
# Mount Google Drive and save
from google.colab import drive
drive.mount('/content/drive')

import shutil
save_dir = '/content/drive/MyDrive/advanced-llm-checkpoints'
os.makedirs(save_dir, exist_ok=True)

shutil.copytree('export', f'{save_dir}/export', dirs_exist_ok=True)
shutil.copy('tokenizer.json', save_dir)
shutil.copytree('configs', f'{save_dir}/configs', dirs_exist_ok=True)
shutil.copytree('checkpoints', f'{save_dir}/checkpoints', dirs_exist_ok=True)

print(f'Saved to Google Drive: {save_dir}')